# **Stacking from scratch**

Berikan penjelasan terkait Stacking!

Source : https://www.geeksforgeeks.org/machine-learning/stacking-in-machine-learning/
Youtube : https://www.youtube.com/watch?v=a4IS1Ai7GCI

In [ ]:
# Import Libraries

**Pendefinisian class stack**

Bertujuan untuk membuat kerangka kerja ensemble learning berbasis stacking dan blending secara modular dan fleksibel. Class ini memungkinkan pengguna untuk menggabungkan beberapa model dasar (base learners) dan satu model meta (final estimator) untuk menghasilkan prediksi akhir yang lebih akurat. Dengan mengatur parameter seperti metode cross-validation, penggunaan blending, serta paralelisasi proses, class ini memberikan kontrol penuh kepada pengguna dalam menerapkan teknik stacking sesuai kebutuhan. Kegunaan utamanya adalah sebagai alat bantu untuk mengimplementasikan ensemble model dari nol (from scratch) tanpa tergantung pada fungsi otomatis dari pustaka seperti scikit-learn, sehingga cocok untuk pembelajaran konsep maupun eksperimen lanjutan dalam machine learning.

Buat class bernama Stack yang berisi <br>

Attribute:


Method:


In [ ]:
# Definisikan class stack

In [1]:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.utils import check_X_y, check_array
from sklearn.utils.validation import check_is_fitted
from joblib import Parallel, delayed

class Stack(BaseEstimator):
    """
    A modular and flexible framework for stacking and blending-based ensemble learning.

    This class allows combining multiple base learners and a meta learner to produce
    more accurate final predictions. It provides control over parameters like
    cross-validation method, blending usage, and parallelization.

    Attributes:
        base_learners (list): A list of base models (estimators) to be used.
        meta_learner: The final estimator (meta model) to combine predictions
                      from the base learners.
        n_splits (int): Number of folds for cross-validation in stacking.
        stack_method (str): Method for generating meta-features ('predict_proba'
                            for classification, 'predict' for regression).
        use_blending (bool): Whether to use blending instead of stacking.
        blending_size (float): The proportion of the training data to use for
                               blending (if use_blending is True).
        random_state (int): Random state for reproducibility.
        n_jobs (int): Number of CPU cores to use for parallel training.
    """

    def __init__(self, base_learners, meta_learner, n_splits=5, stack_method='predict',
                 use_blending=False, blending_size=0.2, random_state=None, n_jobs=1):
        self.base_learners = base_learners
        self.meta_learner = meta_learner
        self.n_splits = n_splits
        self.stack_method = stack_method
        self.use_blending = use_blending
        self.blending_size = blending_size
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.meta_features_ = None
        self.is_fitted_ = False

    def fit(self, X, y):
        """
        Fit the stacking ensemble model.

        Args:
            X (array-like): Training data features.
            y (array-like): Training data target.

        Returns:
            self: Fitted estimator.
        """
        X, y = check_X_y(X, y, accept_sparse=True)

        if self.use_blending:
            return self._fit_blending(X, y)
        else:
            return self._fit_stacking(X, y)

    def _fit_stacking(self, X, y):
        """Fit the stacking model using cross-validation."""
        if self.stack_method not in ['predict', 'predict_proba']:
            raise ValueError("stack_method must be 'predict' or 'predict_proba'")

        meta_features_train = np.zeros((X.shape[0], len(self.base_learners)))

        if hasattr(self.base_learners[0], 'predict_proba') and self.stack_method == 'predict_proba':
            meta_features_train = np.zeros((X.shape[0], len(self.base_learners) * y.nunique())) # Assuming y is a pandas Series or similar with nunique

        if self.stack_method == 'predict_proba' and not all(hasattr(learner, 'predict_proba') for learner in self.base_learners):
             raise AttributeError("All base learners must have 'predict_proba' method for stack_method='predict_proba'")

        if hasattr(self.base_learners[0], 'predict_proba') and self.stack_method == 'predict_proba':
             kf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        else:
            kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)


        def train_base_learner(base_learner, X_train_fold, y_train_fold, X_val_fold):
            learner = base_learner.__class__(**base_learner.get_params())
            learner.fit(X_train_fold, y_train_fold)
            if self.stack_method == 'predict_proba' and hasattr(learner, 'predict_proba'):
                return learner.predict_proba(X_val_fold)
            else:
                return learner.predict(X_val_fold)

        for i, base_learner in enumerate(self.base_learners):
             fold_predictions = Parallel(n_jobs=self.n_jobs)(
                delayed(train_base_learner)(base_learner, X[train_index], y[train_index], X[val_index])
                for train_index, val_index in kf.split(X, y)
            )

             if self.stack_method == 'predict_proba' and hasattr(base_learner, 'predict_proba'):
                # Concatenate predictions from all folds along the row axis
                meta_features_train[:, i*y.nunique() : (i+1)*y.nunique()] = np.concatenate(fold_predictions, axis=0)
             else:
                 meta_features_train[:, i] = np.concatenate(fold_predictions, axis=0)


        self.meta_learner.fit(meta_features_train, y)
        self.is_fitted_ = True
        return self

    def _fit_blending(self, X, y):
        """Fit the blending model."""
        if self.stack_method not in ['predict', 'predict_proba']:
            raise ValueError("stack_method must be 'predict' or 'predict_proba'")

        from sklearn.model_selection import train_test_split
        X_train, X_blend, y_train, y_blend = train_test_split(
            X, y, test_size=self.blending_size, random_state=self.random_state
        )

        for base_learner in self.base_learners:
            base_learner.fit(X_train, y_train)

        meta_features_blend = self._get_meta_features(X_blend)
        self.meta_learner.fit(meta_features_blend, y_blend)
        self.is_fitted_ = True
        return self


    def _get_meta_features(self, X):
        """Generate meta-features from base learner predictions."""
        meta_features = []
        for base_learner in self.base_learners:
            if self.stack_method == 'predict_proba' and hasattr(base_learner, 'predict_proba'):
                meta_features.append(base_learner.predict_proba(X))
            elif self.stack_method == 'predict':
                meta_features.append(base_learner.predict(X).reshape(-1, 1))
            else:
                 raise AttributeError("Base learner does not have 'predict_proba' method for stack_method='predict_proba'")

        return np.hstack(meta_features)

    def predict(self, X):
        """
        Make predictions using the stacking ensemble model.

        Args:
            X (array-like): Data features to make predictions on.

        Returns:
            array: Predicted target values.
        """
        check_is_fitted(self, 'is_fitted_')
        X = check_array(X, accept_sparse=True)

        meta_features_test = self._get_meta_features(X)
        return self.meta_learner.predict(meta_features_test)

    def predict_proba(self, X):
        """
        Make probability predictions using the stacking ensemble model
        (for classification).

        Args:
            X (array-like): Data features to make probability predictions on.

        Returns:
            array: Predicted class probabilities.
        """
        if not hasattr(self.meta_learner, 'predict_proba'):
            raise AttributeError("Meta learner does not have 'predict_proba' method.")

        check_is_fitted(self, 'is_fitted_')
        X = check_array(X, accept_sparse=True)

        meta_features_test = self._get_meta_features(X)
        return self.meta_learner.predict_proba(meta_features_test)

### **Upload datasets**

In [2]:
# Load dataset iris dari sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
iris_df = iris.frame

# Pisahkan fitur dan target
X1 = iris_df.drop(columns=['target'])
y1 = iris_df['target']

# Encode target (optional, untuk case multiclass stacking)
# For demonstration, we will use the original target which is already numeric.
# If you were doing binary classification with stacking and needed one-hot encoding for the meta-features
# you would do it here, but for this example, it's not necessary.


# Split data menjadi train dan test
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, random_state=0)

print("Shape of X1_train:", X1_train.shape)
print("Shape of y1_train:", y1_train.shape)
print("Shape of X1_test:", X1_test.shape)
print("Shape of y1_test:", y1_test.shape)

Shape of X1_train: (112, 4)
Shape of y1_train: (112,)
Shape of X1_test: (38, 4)
Shape of y1_test: (38,)


# **Model training and evaluation of the obtained results**
Both in the case of classification and regression, stacking and blending showed the same and not the best results. As a rule, this situation occurs for two reasons: given that metadata is based on predictions of basic models, the presence of weak basic models can reduce the accuracy of stronger ones which will reduce the final prediction as a whole. Also a small amount of training data often leads to overfitting which in turn reduces the accuracy of predictions.

In this case the problem can be partially solved by setting stack_method='predict_proba' when each basic classifier outputs class membership probabilities instead of the classes themselves which can help increase accuracy in the case of non-mutually exclusive classes. Also this method works better with noise in the data. As you can see this method has significantly increased the accuracy of the model. With the right selection of models and hyperparameters the accuracy will be even higher.

Most often stacking shows slightly better results than blending due to the use of k-fold cross-validation but usually the difference is noticeable only on a large amount of data.

In [ ]:
# StackingClassifier dan Blending Classifier


## Implement blending with custom stack class

### Subtask:
Instantiate the custom `Stack` class with the defined base learners and meta learner, setting `use_blending=True`.

**Reasoning**:
Instantiate the custom Stack class with the defined base learners and meta learner, setting use_blending=True.

In [11]:
# Instantiate the custom Stack class for blending
custom_blending_model = Stack(base_learners=base_learners, meta_learner=meta_learner, use_blending=True, random_state=0)

## Implement stacking with custom stack class

### Subtask:
Instantiate the custom `Stack` class with the defined base learners and meta learner, using the default stacking method (cross-validation).

**Reasoning**:
Instantiate the custom Stack class with the defined base learners and meta learner using default parameters.

In [10]:
# Instantiate the custom Stack class
custom_stacking_model = Stack(base_learners=base_learners, meta_learner=meta_learner)

## Define base learners and meta learner

### Subtask:
Select appropriate base models (e.g., Logistic Regression, Decision Tree, K-Nearest Neighbors) and a meta model (e.g., Logistic Regression) for classification.

**Reasoning**:
Import necessary classifier classes and instantiate base and meta learner models as instructed.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Instantiate base learner models
base_learners = [
    LogisticRegression(random_state=0),
    DecisionTreeClassifier(random_state=0),
    KNeighborsClassifier()
]

# Instantiate a meta learner model
meta_learner = LogisticRegression(random_state=0)

**StackingClassifier (scikit-learn)**


* Merupakan implementasi resmi stacking untuk klasifikasi dalam scikit-learn.

* Mempermudah proses ensemble dengan base learners dan final estimator (meta learner) dalam satu objek.

* Sudah menangani cross-validation secara internal sehingga mengurangi risiko data leakage.

* Mendukung parameter passthrough=True jika ingin menggabungkan fitur asli dengan meta-features.

* Cocok untuk digunakan dalam pipeline dan produksi karena stabil dan teruji.

In [ ]:
# StackingClassifier (scikit-learn)

# sk_estimators

# stack_methods

for i, method in enumerate(stack_methods):

## Summary:

### Data Analysis Key Findings

* Instances of three base learner models (Logistic Regression, Decision Tree, and K-Nearest Neighbors) and a Logistic Regression meta learner were successfully created.
* Custom `Stack` class instances were successfully created for both stacking (default) and blending (`use_blending=True`).
* Scikit-learn's `StackingClassifier` was successfully instantiated using the same base and meta learners.
* All three models (custom stacking, custom blending, and scikit-learn stacking) were trained on the training data and evaluated on the test data.
* The evaluation metrics (accuracy, precision, recall, and F1-score) showed identical performance for all three implemented ensemble models on the test dataset.

### Insights or Next Steps

* The identical performance suggests that for this specific dataset and choice of base/meta-learners, the benefits of blending over stacking (or vice versa) were not realized, and the custom implementations performed equivalently to the scikit-learn version.
* Further exploration could involve using different base learners, meta learners, or datasets to see if performance differences emerge between stacking, blending, and their implementations.

## Compare results

### Subtask:
Present the evaluation results for all models to compare their performance.

**Reasoning**:
Display the metrics_df DataFrame and interpret the results as requested by the subtask.

In [14]:
# Display the metrics_df DataFrame
display(metrics_df)

# Interpret the results
print("Evaluation Results:")
print("The DataFrame above shows the performance metrics (accuracy, precision, recall, and F1-score) for the Custom Stacking, Custom Blending, and Scikit-learn Stacking models.")
print("For this specific dataset and choice of base/meta learners, all three models achieved identical results across all metrics.")
print("This indicates that both the custom implementations and the scikit-learn implementation of stacking performed equally well on the test data.")
print("Blending also showed the same performance as stacking in this particular scenario.")

,accuracy,precision,recall,f1_score
Custom Stacking,0.973684,0.976316,0.973684,0.973952
Custom Blending,0.973684,0.976316,0.973684,0.973952
Scikit-learn Stacking,0.973684,0.976316,0.973684,0.973952


Evaluation Results:
The DataFrame above shows the performance metrics (accuracy, precision, recall, and F1-score) for the Custom Stacking, Custom Blending, and Scikit-learn Stacking models.
For this specific dataset and choice of base/meta learners, all three models achieved identical results across all metrics.
This indicates that both the custom implementations and the scikit-learn implementation of stacking performed equally well on the test data.
Blending also showed the same performance as stacking in this particular scenario.


## Train and evaluate models

### Subtask:
Train all three models (custom stacking, custom blending, scikit-learn stacking) on the training data and evaluate their performance on the test data using appropriate classification metrics (e.g., accuracy, precision, recall, F1-score).

**Reasoning**:
Train the custom stacking, custom blending, and scikit-learn stacking models, and then evaluate their performance using classification metrics.

In [13]:
# Train the custom stacking model
custom_stacking_model.fit(X1_train, y1_train)

# Train the custom blending model
custom_blending_model.fit(X1_train, y1_train)

# Train the scikit-learn stacking model
sklearn_stacking_model.fit(X1_train, y1_train)

# Import necessary metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test data
custom_stacking_pred = custom_stacking_model.predict(X1_test)
custom_blending_pred = custom_blending_model.predict(X1_test)
sklearn_stacking_pred = sklearn_stacking_model.predict(X1_test)

# Calculate metrics for each model
metrics = {}

metrics['Custom Stacking'] = {
    'accuracy': accuracy_score(y1_test, custom_stacking_pred),
    'precision': precision_score(y1_test, custom_stacking_pred, average='weighted'),
    'recall': recall_score(y1_test, custom_stacking_pred, average='weighted'),
    'f1_score': f1_score(y1_test, custom_stacking_pred, average='weighted')
}

metrics['Custom Blending'] = {
    'accuracy': accuracy_score(y1_test, custom_blending_pred),
    'precision': precision_score(y1_test, custom_blending_pred, average='weighted'),
    'recall': recall_score(y1_test, custom_blending_pred, average='weighted'),
    'f1_score': f1_score(y1_test, custom_blending_pred, average='weighted')
}

metrics['Scikit-learn Stacking'] = {
    'accuracy': accuracy_score(y1_test, sklearn_stacking_pred),
    'precision': precision_score(y1_test, sklearn_stacking_pred, average='weighted'),
    'recall': recall_score(y1_test, sklearn_stacking_pred, average='weighted'),
    'f1_score': f1_score(y1_test, sklearn_stacking_pred, average='weighted')
}

# Display the metrics
import pandas as pd
metrics_df = pd.DataFrame(metrics).T
display(metrics_df)

,accuracy,precision,recall,f1_score
Custom Stacking,0.973684,0.976316,0.973684,0.973952
Custom Blending,0.973684,0.976316,0.973684,0.973952
Scikit-learn Stacking,0.973684,0.976316,0.973684,0.973952


## Implement stacking with scikit-learn stackingclassifier

### Subtask:
Instantiate `StackingClassifier` from scikit-learn with the same base learners and meta learner.

**Reasoning**:
Import the StackingClassifier, create a list of base learners with names, and instantiate the StackingClassifier.

In [12]:
from sklearn.ensemble import StackingClassifier

# Create a list of tuples for scikit-learn's StackingClassifier
sk_estimators = [
    ('lr', base_learners[0]),
    ('dt', base_learners[1]),
    ('knn', base_learners[2])
]

# Instantiate the StackingClassifier
sklearn_stacking_model = StackingClassifier(
    estimators=sk_estimators,
    final_estimator=meta_learner,
    cv=5 # Use the same number of folds as in the custom class default
)

# Task
Generate Google Colab code to implement stacking and blending classifiers using both a custom class and scikit-learn's `StackingClassifier`.

## Define base learners and meta learner

### Subtask:
Select appropriate base models (e.g., Logistic Regression, Decision Tree, K-Nearest Neighbors) and a meta model (e.g., Logistic Regression) for classification.


**Reasoning**:
Import necessary classifier classes and instantiate base and meta learner models as instructed.



In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Instantiate base learner models
base_learners = [
    LogisticRegression(random_state=0),
    DecisionTreeClassifier(random_state=0),
    KNeighborsClassifier()
]

# Instantiate a meta learner model
meta_learner = LogisticRegression(random_state=0)

## Implement stacking with custom stack class

### Subtask:
Instantiate the custom `Stack` class with the defined base learners and meta learner, using the default stacking method (cross-validation).


**Reasoning**:
Instantiate the custom Stack class with the defined base learners and meta learner using default parameters.



In [4]:
# Instantiate the custom Stack class
custom_stacking_model = Stack(base_learners=base_learners, meta_learner=meta_learner)

## Implement blending with custom stack class

### Subtask:
Instantiate the custom `Stack` class with the defined base learners and meta learner, setting `use_blending=True`.


**Reasoning**:
Instantiate the custom Stack class with the defined base learners and meta learner, setting use_blending=True.



In [5]:
# Instantiate the custom Stack class for blending
custom_blending_model = Stack(base_learners=base_learners, meta_learner=meta_learner, use_blending=True, random_state=0)

## Implement stacking with scikit-learn stackingclassifier

### Subtask:
Instantiate `StackingClassifier` from scikit-learn with the same base learners and meta learner.


**Reasoning**:
Import the StackingClassifier, create a list of base learners with names, and instantiate the StackingClassifier.



In [6]:
from sklearn.ensemble import StackingClassifier

# Create a list of tuples for scikit-learn's StackingClassifier
sk_estimators = [
    ('lr', base_learners[0]),
    ('dt', base_learners[1]),
    ('knn', base_learners[2])
]

# Instantiate the StackingClassifier
sklearn_stacking_model = StackingClassifier(
    estimators=sk_estimators,
    final_estimator=meta_learner,
    cv=5 # Use the same number of folds as in the custom class default
)

## Train and evaluate models

### Subtask:
Train all three models (custom stacking, custom blending, scikit-learn stacking) on the training data and evaluate their performance on the test data using appropriate classification metrics (e.g., accuracy, precision, recall, F1-score).


**Reasoning**:
Train the custom stacking, custom blending, and scikit-learn stacking models, and then evaluate their performance using classification metrics.



In [7]:
# Train the custom stacking model
custom_stacking_model.fit(X1_train, y1_train)

# Train the custom blending model
custom_blending_model.fit(X1_train, y1_train)

# Train the scikit-learn stacking model
sklearn_stacking_model.fit(X1_train, y1_train)

# Import necessary metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test data
custom_stacking_pred = custom_stacking_model.predict(X1_test)
custom_blending_pred = custom_blending_model.predict(X1_test)
sklearn_stacking_pred = sklearn_stacking_model.predict(X1_test)

# Calculate metrics for each model
metrics = {}

metrics['Custom Stacking'] = {
    'accuracy': accuracy_score(y1_test, custom_stacking_pred),
    'precision': precision_score(y1_test, custom_stacking_pred, average='weighted'),
    'recall': recall_score(y1_test, custom_stacking_pred, average='weighted'),
    'f1_score': f1_score(y1_test, custom_stacking_pred, average='weighted')
}

metrics['Custom Blending'] = {
    'accuracy': accuracy_score(y1_test, custom_blending_pred),
    'precision': precision_score(y1_test, custom_blending_pred, average='weighted'),
    'recall': recall_score(y1_test, custom_blending_pred, average='weighted'),
    'f1_score': f1_score(y1_test, custom_blending_pred, average='weighted')
}

metrics['Scikit-learn Stacking'] = {
    'accuracy': accuracy_score(y1_test, sklearn_stacking_pred),
    'precision': precision_score(y1_test, sklearn_stacking_pred, average='weighted'),
    'recall': recall_score(y1_test, sklearn_stacking_pred, average='weighted'),
    'f1_score': f1_score(y1_test, sklearn_stacking_pred, average='weighted')
}

# Display the metrics
import pandas as pd
metrics_df = pd.DataFrame(metrics).T
display(metrics_df)

,accuracy,precision,recall,f1_score
Custom Stacking,0.973684,0.976316,0.973684,0.973952
Custom Blending,0.973684,0.976316,0.973684,0.973952
Scikit-learn Stacking,0.973684,0.976316,0.973684,0.973952


## Compare results

### Subtask:
Present the evaluation results for all models to compare their performance.


**Reasoning**:
Display the metrics_df DataFrame and interpret the results as requested by the subtask.



In [8]:
# Display the metrics_df DataFrame
display(metrics_df)

# Interpret the results
print("Evaluation Results:")
print("The DataFrame above shows the performance metrics (accuracy, precision, recall, and F1-score) for the Custom Stacking, Custom Blending, and Scikit-learn Stacking models.")
print("For this specific dataset and choice of base/meta learners, all three models achieved identical results across all metrics.")
print("This indicates that both the custom implementations and the scikit-learn implementation of stacking performed equally well on the test data.")
print("Blending also showed the same performance as stacking in this particular scenario.")

,accuracy,precision,recall,f1_score
Custom Stacking,0.973684,0.976316,0.973684,0.973952
Custom Blending,0.973684,0.976316,0.973684,0.973952
Scikit-learn Stacking,0.973684,0.976316,0.973684,0.973952


Evaluation Results:
The DataFrame above shows the performance metrics (accuracy, precision, recall, and F1-score) for the Custom Stacking, Custom Blending, and Scikit-learn Stacking models.
For this specific dataset and choice of base/meta learners, all three models achieved identical results across all metrics.
This indicates that both the custom implementations and the scikit-learn implementation of stacking performed equally well on the test data.
Blending also showed the same performance as stacking in this particular scenario.


## Summary:

### Data Analysis Key Findings

*   Instances of three base learner models (Logistic Regression, Decision Tree, and K-Nearest Neighbors) and a Logistic Regression meta learner were successfully created.
*   Custom `Stack` class instances were successfully created for both stacking (default) and blending (`use_blending=True`).
*   Scikit-learn's `StackingClassifier` was successfully instantiated using the same base and meta learners.
*   All three models (custom stacking, custom blending, and scikit-learn stacking) were trained on the training data and evaluated on the test data.
*   The evaluation metrics (accuracy, precision, recall, and F1-score) showed identical performance for all three implemented ensemble models on the test dataset.

### Insights or Next Steps

*   The identical performance suggests that for this specific dataset and choice of base/meta-learners, the benefits of blending over stacking (or vice versa) were not realized, and the custom implementations performed equivalently to the scikit-learn version.
*   Further exploration could involve using different base learners, meta learners, or datasets to see if performance differences emerge between stacking, blending, and their implementations.
